# Session 3: one solver for any 2D pin-jointed truss
**A. Real-world problem.** Statics (Session 2) fails when a truss has extra bars, and it
says nothing about how far the truss sags. A footbridge must be strong *and* stiff. We
build the method that computer programs use: the direct stiffness method, in which each
bar is a spring and the whole truss is one equation K u = f.

**Plan.** (1) one bar as a spring, (2) assemble all bars into K, (3) apply supports and
solve, (4) recover bar forces, (5) validate against hand results, equilibrium and a
virtual-work formula, (6) compare Pratt, Howe and Warren, (7) test a missing bar,
stiffness changes and Euler buckling.

Educational model: ideal pins, small deflections, straight weightless bars. Not for real
structures.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: one bar is a spring along its own axis
A bar from joint i to joint k has length L and axial stiffness EA/L (E = Young's modulus,
A = cross-section area). With direction cosines c = cos(angle), s = sin(angle), the bar
contributes the 4 x 4 matrix (EA/L) * [[c c, c s, -c c, -c s], [c s, s s, -c s, -s s],
[-c c, -c s, c c, c s], [-c s, -s s, c s, s s]] to the joints it touches.
Units: lengths in m, forces in kN, EA in kN, displacements in m.

In [ ]:
def member_geometry(nodes, members):
    """Lengths and direction cosines (cx, cy) of every bar."""
    nodes = np.asarray(nodes, float)
    vec = np.array([nodes[k] - nodes[i] for i, k in members])
    length = np.linalg.norm(vec, axis=1)
    return length, vec / length[:, None]


def assemble_k(nodes, members, EA):
    """Global stiffness matrix (2 j x 2 j). EA is one number or one value per bar (kN)."""
    nodes = np.asarray(nodes, float)
    EA = np.broadcast_to(np.asarray(EA, float), (len(members),))
    length, cs = member_geometry(nodes, members)
    K = np.zeros((2 * len(nodes), 2 * len(nodes)))
    for e, (i, k) in enumerate(members):
        c, s = cs[e]
        blk = np.array([[c * c, c * s], [c * s, s * s]])
        dofs = [2 * i, 2 * i + 1, 2 * k, 2 * k + 1]
        ke = EA[e] / length[e] * np.block([[blk, -blk], [-blk, blk]])
        K[np.ix_(dofs, dofs)] += ke
    return K

## Step 2: supports, the solve, and the recovery of forces
Supports remove unknowns: we keep only the free degrees of freedom (dofs), solve
K_ff u_f = f_f, and then recover each bar force from its stretch:
F = (EA/L) * ((u_k - u_i) . (c, s)). Positive F = tension. Reactions follow from
R = K u - f at the fixed dofs.
A truss that is a *mechanism* has a singular K_ff: some motion costs no energy. We detect
it from the rank of K_ff (or a huge condition number).

In [ ]:
def free_dofs(n_nodes, supports):
    fixed = {2 * n + d for n, fl in supports.items() for d in (0, 1) if fl[d]}
    return [d for d in range(2 * n_nodes) if d not in fixed], sorted(fixed)


def is_mechanism(K, free):
    """True when the free-dof stiffness matrix is rank deficient (the truss can move freely)."""
    Kff = K[np.ix_(free, free)]
    return bool(np.linalg.matrix_rank(Kff) < len(free))


def solve_truss(nodes, members, EA, supports, loads):
    """Direct stiffness solution. Returns dict with u (j, 2) in m, forces (kN, tension +), reactions (2j,) in kN."""
    nodes = np.asarray(nodes, float)
    K = assemble_k(nodes, members, EA)
    free, fixed = free_dofs(len(nodes), supports)
    if is_mechanism(K, free):
        raise ValueError("mechanism: the stiffness matrix is singular (a bar is missing or supports are too few)")
    f = np.zeros(2 * len(nodes))
    for n, (fx, fy) in loads.items():
        f[2 * n] += fx
        f[2 * n + 1] += fy
    u = np.zeros(2 * len(nodes))
    u[free] = np.linalg.solve(K[np.ix_(free, free)], f[free])
    length, cs = member_geometry(nodes, members)
    EAv = np.broadcast_to(np.asarray(EA, float), (len(members),))
    forces = np.array([EAv[e] / length[e] * (u[2 * k:2 * k + 2] - u[2 * i:2 * i + 2]) @ cs[e]
                       for e, (i, k) in enumerate(members)])
    R = np.zeros(2 * len(nodes))
    R[fixed] = (K @ u - f)[fixed]
    return {"u": u.reshape(-1, 2), "forces": forces, "reactions": R}


def joint_residuals(nodes, members, forces, reactions, loads):
    """Net force on every joint (j, 2); all zeros when every pin is in equilibrium."""
    nodes = np.asarray(nodes, float)
    _, cs = member_geometry(nodes, members)
    res = np.array(reactions, float).reshape(-1, 2)
    for e, (i, k) in enumerate(members):
        res[i] += forces[e] * cs[e]
        res[k] -= forces[e] * cs[e]
    for n, (fx, fy) in loads.items():
        res[n] += (fx, fy)
    return res

## Step 3: three bridge shapes under the same load
Each is 8 m long with 4 panels of 2 m and a height of 2 m. Bottom joints sit on the deck;
a pin at the left foot, a roller at the right foot, and a 10 kN load at midspan.
Pratt: diagonals slope down towards the middle. Howe: diagonals slope up towards the
middle. Warren: only sloping bars, alternately up and down.

In [ ]:
def bridge(kind, span=8.0, panels=4, height=2.0):
    """Return nodes, members for 'pratt', 'howe' or 'warren' (panels must be even)."""
    dx = span / panels
    bottom = [[dx * i, 0.0] for i in range(panels + 1)]
    mem = [(i, i + 1) for i in range(panels)]
    if kind == "warren":
        top = [[dx * (i + 0.5), height] for i in range(panels)]
        nb = panels + 1
        mem += [(nb + i, nb + i + 1) for i in range(panels - 1)]
        for i in range(panels):
            mem += [(i, nb + i), (nb + i, i + 1)]
        return np.array(bottom + top, float), mem
    top = [[dx * i, height] for i in range(1, panels)]
    nb = panels + 1
    t = lambda i: nb + i - 1                       # index of the top joint above bottom joint i
    mem += [(t(i), t(i + 1)) for i in range(1, panels - 1)]
    mem += [(i, t(i)) for i in range(1, panels)]   # verticals
    mem += [(0, t(1)), (panels, t(panels - 1))]    # inclined end posts
    half = panels // 2
    for i in range(1, half):
        if kind == "pratt":
            mem += [(t(i), i + 1), (t(panels - i), panels - i - 1)]
        else:
            mem += [(i, t(i + 1)), (panels - i, t(panels - i - 1))]
    return np.array(bottom + top, float), mem


def standard_case(kind, load=10.0, EA=80000.0, **kw):
    nodes, members = bridge(kind, **kw)
    panels = int(round(nodes[:, 0].max() / 2.0))
    n_bottom = len([1 for x, y in nodes if y == 0.0])
    supports = {0: (1, 1), n_bottom - 1: (0, 1)}
    mid = n_bottom // 2
    return nodes, members, supports, {mid: (0.0, -load)}, EA


def tension_members(members, forces, tol=1e-6):
    return [m for m, f in zip(members, forces) if f > tol]

## Step 4: an independent check of the deflection, by virtual work
Unit-load (virtual work) formula: the deflection of a joint in the direction of a unit
load placed there is delta = sum over bars of F_i f_i L_i / (EA)_i, where F_i are the
bar forces for the real load and f_i those for a unit load at that joint.
Simple case (the Session 1 truss): two sloping bars of length Ls and a bottom tie of
length L, load P at the apex. Then F = -P/(2 sin t), f = -1/(2 sin t) for the sloping
bars, F = P/(2 tan t), f = 1/(2 tan t) for the tie.

In [ ]:
def virtual_work_deflection(forces, unit_forces, length, EA):
    return float(np.sum(forces * unit_forces * length / EA))


def triangle_apex_deflection(P, span, height, EA):
    """Closed-form apex deflection (m, downwards) of the two-bar-plus-tie truss."""
    th = np.arctan2(height, span / 2.0)
    Ls = np.hypot(span / 2.0, height)
    top = 2 * (P / (2 * np.sin(th))) * (1 / (2 * np.sin(th))) * Ls / EA
    tie = (P / (2 * np.tan(th))) * (1 / (2 * np.tan(th))) * span / EA
    return float(top + tie)

## Step 5: when a bar goes missing, and when a bar buckles
Remove one diagonal and the stiffness matrix becomes singular: the truss is a mechanism.
For a compression bar the Euler load for a pin-ended column is P_cr = pi^2 E I / L^2
(I = second moment of area). Used here only as a check that a compression bar does not
fail by sideways bending. For a solid round bar of area A, I = A^2 / (4 pi).

In [ ]:
def euler_load_kn(E_gpa, area_mm2, length_m):
    """Euler buckling load (kN) of a pin-ended solid round bar."""
    I_m4 = (area_mm2 * 1e-6) ** 2 / (4 * np.pi)
    return float(np.pi ** 2 * E_gpa * 1e6 * I_m4 / length_m ** 2)

## Step 6: size a footbridge (mini-project tools)
Fully stressed sizing: every bar gets the area |F| / sigma_allow, but not less than a
minimum area. If the sag is above the limit, all areas are scaled up together (for a
determinate truss the sag is proportional to 1/area). Weight = density * sum(A L).
Illustrative numbers only.

In [ ]:
def size_truss(nodes, members, supports, loads, sigma_mpa=150.0, a_min_mm2=100.0, E_gpa=200.0, limit_mm=None):
    """Return dict(area, weight_kg, sag_mm, scaled) for a determinate truss."""
    length, _ = member_geometry(nodes, members)
    f0 = solve_truss(nodes, members, E_gpa * 300.0, supports, loads)["forces"]
    area = np.maximum(np.abs(f0) * 1000.0 / sigma_mpa, a_min_mm2)
    scaled = 1.0
    sag = -solve_truss(nodes, members, E_gpa * area, supports, loads)["u"][:, 1].min() * 1000.0
    if limit_mm is not None and sag > limit_mm:
        scaled = sag / limit_mm
        area = area * scaled
        sag = limit_mm
    weight = float(7850.0 * np.sum(area * 1e-6 * length))
    return {"area": area, "weight_kg": weight, "sag_mm": float(sag), "scaled": float(scaled)}


def footbridge_case(kind, line_load=6.0):
    """12 m footbridge, 6 panels of 2 m, height 2 m, uniform deck load line_load (kN/m)."""
    nodes, members = bridge(kind, span=12.0, panels=6, height=2.0)
    n_bottom = 7
    supports = {0: (1, 1), n_bottom - 1: (0, 1)}
    loads = {i: (0.0, -line_load * 2.0) for i in range(1, n_bottom - 1)}
    return nodes, members, supports, loads

## Step 7: see it

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
    for ax, kind in zip(axs, ("pratt", "howe", "warren")):
        nodes, members, supports, loads, EA = standard_case(kind)
        r = solve_truss(nodes, members, EA, supports, loads)
        F, u = r["forces"], r["u"]
        big = np.abs(F).max()
        for (i, k), fk in zip(members, F):
            col = "tab:blue" if fk > 1e-6 else ("tab:red" if fk < -1e-6 else "0.6")
            ax.plot(nodes[[i, k], 0], nodes[[i, k], 1], color=col, lw=1 + 5 * abs(fk) / big)
        d = nodes + 200 * u
        for (i, k) in members:
            ax.plot(d[[i, k], 0], d[[i, k], 1], "k--", lw=0.7)
        ax.set_aspect("equal")
        ax.set_title(f"{kind}: max sag {-u[:, 1].min() * 1000:.2f} mm (drawn x200)", fontsize=9)
        ax.set_xlabel("x (m)")
        ax.set_ylabel("y (m)")
        ax.set_ylim(-0.8, 2.6)
        nt = len(tension_members(members, F))
        print(f"{kind:7s}: {len(members)} bars, {nt} in tension, "
              f"max tension {F.max():+.2f} kN, max compression {F.min():+.2f} kN, "
              f"sag {-u[:, 1].min() * 1000:.3f} mm")
    fig.tight_layout()
    fig.savefig(OUT / "three_trusses.png", dpi=120)

    # sensitivity of sag to member stiffness (determinate trusses: forces do not change)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    eas = np.linspace(40000, 200000, 40)
    for kind in ("pratt", "howe", "warren"):
        nodes, members, supports, loads, _ = standard_case(kind)
        sag = [-solve_truss(nodes, members, ea, supports, loads)["u"][:, 1].min() * 1000 for ea in eas]
        ax2.plot(eas / 1000, sag, label=kind)
    ax2.set_xlabel("bar stiffness EA (MN)")
    ax2.set_ylabel("largest sag (mm)")
    ax2.set_title("Sag is inversely proportional to EA")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "sag_vs_ea.png", dpi=120)

    nodes, members, supports, loads, EA = standard_case("pratt")
    try:
        solve_truss(nodes, members[:-1], EA, supports, loads)
    except ValueError as err:
        print("missing bar ->", err)
    A = 300.0
    print("Euler load of a 2 m long solid bar of 300 mm^2:", round(euler_load_kn(200.0, A, 2.0), 3), "kN")
    for kind in ("pratt", "howe", "warren"):
        nodes, members, supports, loads = footbridge_case(kind)
        r = size_truss(nodes, members, supports, loads, limit_mm=24.0)
        print(f"footbridge {kind:7s}: weight {r['weight_kg']:.0f} kg, sag {r['sag_mm']:.1f} mm, scale {r['scaled']:.2f}")
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print the bar forces of the Pratt truss for `EA = 40000` and for `EA = 160000` and confirm they are identical while the sag changes.
2. **Stretch.** Add the extra bar `(1, 6)` to the Pratt truss and plot its force against its own EA; compare with the values -2.17, -2.80 and -3.28 kN for 0.5, 1 and 2 times EA.
3. **Explain.** Explain why the stiffness solver can handle the indeterminate truss while the pin-by-pin solver of Session 2 cannot.

## Assessment
- **A. Conceptual.** In the Pratt truss the diagonals are in tension and in the Howe truss they are in compression, although both carry the same load. Why does it matter for the design which bars are pulled?
- **B. Numerical.** Use the unit-load method for the three-bar triangle truss (span 6 m, height 3 m, 10 kN at the apex, EA = 80 000 kN). Give the apex sag in mm, then check with the solver.
- **C. Graph and simulation interpretation.** On the sag against EA graph, why are the three curves hyperbolas (proportional to 1/EA), and what does that tell you about doubling the steel area?
- **D. Design and optimisation challenge.** Design: the 12 m footbridge (6 panels, 2 m high, 6 kN/m) is sized by stress alone (150 MPa, 100 mm^2 minimum). The sag limit is 15 mm. For each shape say whether the stress-only design passes, and what the lightest shape is when the limit is enforced by scaling all areas up together.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # --- stiffness matrix basics
    nodes1 = np.array([[0.0, 0.0], [2.0, 0.0]])
    K1 = L.assemble_k(nodes1, [(0, 1)], 1000.0)
    checks.append(("one horizontal bar is a spring of stiffness EA/L = 500 kN/m", abs(K1[0, 0] - 500.0) < 1e-9 and abs(K1[0, 2] + 500.0) < 1e-9))
    n_p, m_p, s_p, l_p, EA = L.standard_case("pratt")
    K = L.assemble_k(n_p, m_p, EA)
    checks.append(("K is symmetric", np.allclose(K, K.T)))
    checks.append(("moving the whole truss sideways costs no force (K times a rigid shift is zero)",
                   np.allclose(K @ np.tile([1.0, 0.0], len(n_p)), 0.0, atol=1e-8)))

    # --- hand-solved triangle truss (Session 1 geometry), method of joints
    tri = np.array([[0.0, 0.0], [6.0, 0.0], [3.0, 3.0]])
    r = L.solve_truss(tri, [(0, 2), (2, 1), (0, 1)], 80000.0, {0: (1, 1), 1: (0, 1)}, {2: (0.0, -10.0)})
    checks.append(("triangle truss: top bars -7.07 kN and bottom bar +5.00 kN, as by hand",
                   np.allclose(r["forces"], [-7.0711, -7.0711, 5.0], atol=1e-3)))
    checks.append(("triangle truss: reactions are 5 kN up at each foot",
                   abs(r["reactions"][1] - 5.0) < 1e-9 and abs(r["reactions"][3] - 5.0) < 1e-9 and abs(r["reactions"][0]) < 1e-9))
    d_formula = L.triangle_apex_deflection(10.0, 6.0, 3.0, 80000.0)
    checks.append(("triangle truss apex sag by the solver equals the virtual-work formula (0.718 mm)",
                   abs(-r["u"][2, 1] - d_formula) < 1e-12 and abs(d_formula * 1000 - 0.718) < 0.0005))

    # --- Pratt, Howe, Warren under the same 10 kN load
    res = {}
    for kind in ("pratt", "howe", "warren"):
        nodes, members, sup, loads, EA = L.standard_case(kind)
        out = L.solve_truss(nodes, members, EA, sup, loads)
        res[kind] = (nodes, members, sup, loads, EA, out)
    ok_eq = all(np.allclose(L.joint_residuals(v[0], v[1], v[5]["forces"], v[5]["reactions"], v[3]), 0.0, atol=1e-9) for v in res.values())
    checks.append(("equilibrium at every joint (all three trusses): residual below 1e-9 kN", ok_eq))
    checks.append(("reactions add up to the 10 kN load (vertical sum)",
                   all(abs(v[5]["reactions"][1::2].sum() - 10.0) < 1e-9 for v in res.values())))

    nodes, members, sup, loads, EA, out = res["pratt"]
    F = out["forces"]
    checks.append(("Pratt by hand: end posts -7.07, diagonals +7.07, top chord -10.00, bottom chords +5.00 kN",
                   np.allclose(F[[9, 10]], -7.0711, atol=1e-3) and np.allclose(F[[11, 12]], 7.0711, atol=1e-3)
                   and np.allclose(F[[4, 5]], -10.0, atol=1e-9) and np.allclose(F[:4], 5.0, atol=1e-9)))
    checks.append(("Pratt: the three verticals carry zero force under this load", np.allclose(F[6:9], 0.0, atol=1e-9)))
    Fh = res["howe"][5]["forces"]
    checks.append(("Howe: diagonals are in compression (-7.07 kN) and the midspan vertical is in tension (+10 kN)",
                   np.allclose(Fh[[11, 12]], -7.0711, atol=1e-3) and abs(Fh[7] - 10.0) < 1e-9))
    Fw = res["warren"][5]["forces"]
    checks.append(("Warren: sloping bars alternate +-5.59 kN (5 / sin 63.4 degrees) and the middle top chord is -10.00 kN",
                   np.allclose(np.abs(Fw[7:]), 5.5902, atol=1e-3) and abs(Fw[5] + 10.0) < 1e-9))
    counts = {k: len(L.tension_members(v[1], v[5]["forces"])) for k, v in res.items()}
    checks.append(("bars in tension: Pratt 6, Howe 7, Warren 8", counts == {"pratt": 6, "howe": 7, "warren": 8}))
    sag = {k: -v[5]["u"][:, 1].min() * 1000.0 for k, v in res.items()}
    checks.append(("midspan sag (mm): Pratt 1.457, Howe 1.832, Warren 1.386; Warren is stiffest",
                   abs(sag["pratt"] - 1.457) < 0.0005 and abs(sag["howe"] - 1.832) < 0.0005 and abs(sag["warren"] - 1.386) < 0.0005
                   and sag["warren"] < sag["pratt"] < sag["howe"]))

    checks.append(("bars and joints: Pratt 13 and 8, Howe 13 and 8, Warren 15 and 9",
                   [len(res[k][1]) for k in res] == [13, 13, 15] and [len(res[k][0]) for k in res] == [8, 8, 9]))
    checks.append(("extreme forces (kN): Pratt +7.07 / -10.00, Howe +10.00 / -7.07, Warren +7.50 / -10.00",
                   np.allclose([res[k][5]["forces"].max() for k in res], [7.0711, 10.0, 7.5], atol=1e-3)
                   and np.allclose([res[k][5]["forces"].min() for k in res], [-10.0, -7.0711, -10.0], atol=1e-3)))
    checks.append(("worked example: doubling the EA of one bottom bar lowers the Pratt sag from 1.457 to 1.426 mm",
                   abs(-L.solve_truss(nodes, members, np.where(np.arange(len(members)) == 0, 2 * EA, EA), sup, loads)["u"][:, 1].min() * 1000 - 1.426) < 0.0005))
    checks.append(("failure case: the stress-sized bar buckles at under 2 percent of the 10 kN load", L.euler_load_kn(200.0, 10000.0 / 150.0, 2.0) / 10.0 < 0.02))
    checks.append(("footbridge trusses have 21, 21 and 23 bars", [len(L.footbridge_case(k)[1]) for k in ("pratt", "howe", "warren")] == [21, 21, 23]))
    # --- worked example: virtual work on the Pratt truss
    length, _ = L.member_geometry(nodes, members)
    sum_F2L = float(np.sum(F * F * length))
    unit = L.solve_truss(nodes, members, EA, sup, {2: (0.0, -1.0)})["forces"]
    vw = L.virtual_work_deflection(F, unit, length, EA)
    checks.append(("worked example: Pratt midspan sag by virtual work, sum(F^2 L) = 1165.69 kN^2 m, gives 1.457 mm like the solver",
                   abs(sum_F2L - 1165.69) < 0.005 and abs(vw * 1000 - 1.457) < 0.0005 and abs(vw + out["u"][2, 1]) < 1e-12
                   and abs(sum_F2L / 10.0 / EA * 1000 - 1.457) < 0.0005))

    # --- mechanism: a missing bar
    free = L.free_dofs(len(nodes), sup)[0]
    K_ok = L.assemble_k(nodes, members, EA)[np.ix_(free, free)]
    K_bad = L.assemble_k(nodes, members[:11] + members[12:], EA)[np.ix_(free, free)]
    raised = False
    try:
        L.solve_truss(nodes, members[:11] + members[12:], EA, sup, loads)
    except ValueError:
        raised = True
    checks.append(("missing diagonal: K is rank deficient, condition number above 1e12, solver refuses",
                   np.linalg.matrix_rank(K_bad) < len(free) and np.linalg.cond(K_bad) > 1e12 and raised
                   and np.linalg.matrix_rank(K_ok) == len(free) and np.linalg.cond(K_ok) < 1e3))

    # --- sensitivity to member stiffness
    ea2 = np.full(len(members), EA)
    ea2[0] *= 2.0
    o2 = L.solve_truss(nodes, members, ea2, sup, loads)
    checks.append(("determinate truss: doubling one bar's EA leaves every force unchanged but lowers the sag",
                   np.allclose(o2["forces"], F, atol=1e-9) and -o2["u"][:, 1].min() < -out["u"][:, 1].min()))
    o_half = L.solve_truss(nodes, members, EA * 2.0, sup, loads)
    checks.append(("doubling EA in every bar halves the sag", abs(-o_half["u"][:, 1].min() * 2 + out["u"][:, 1].min()) < 1e-12))
    m2 = members + [(1, 6)]
    extra = []
    for fac in (0.5, 1.0, 2.0):
        e = np.full(len(m2), EA)
        e[-1] *= fac
        extra.append(L.solve_truss(nodes, m2, e, sup, loads)["forces"][-1])
    checks.append(("indeterminate (m + r = 17 > 2 j = 16): the extra bar's force depends on its EA: -2.17, -2.80, -3.28 kN",
                   len(m2) + 3 == 17 and 2 * len(nodes) == 16 and np.allclose(extra, [-2.168, -2.803, -3.284], atol=0.0006)))

    # --- failure case: Euler buckling of the top chord, sized on stress only
    area = 10.0 * 1000.0 / 150.0
    d = np.sqrt(4 * area / np.pi)
    I_hand = np.pi * d ** 4 / 64.0                       # mm^4
    pcr_hand = np.pi ** 2 * 200e3 * I_hand / 2000.0 ** 2 / 1000.0   # N/mm^2 * mm^4 / mm^2 -> N -> kN
    pcr = L.euler_load_kn(200.0, area, 2.0)
    checks.append(("failure case: a 66.7 mm^2 round bar (stress-sized for 10 kN, 2 m long) has P_cr = 0.175 kN, far below 10 kN",
                   abs(pcr - pcr_hand) < 1e-9 and abs(pcr - 0.1745) < 0.0005 and pcr < 10.0))
    checks.append(("Euler load falls with the square of length: doubling L divides P_cr by 4",
                   abs(L.euler_load_kn(200.0, 300.0, 4.0) * 4 - L.euler_load_kn(200.0, 300.0, 2.0)) < 1e-12))

    # --- mini-project: 12 m footbridge, 6 kN/m, 150 MPa, 100 mm^2 minimum, sag limit 15 mm
    w, sg, sc = {}, {}, {}
    for kind in ("pratt", "howe", "warren"):
        n, m, s, lo = L.footbridge_case(kind)
        free_sag = L.size_truss(n, m, s, lo)
        lim = L.size_truss(n, m, s, lo, limit_mm=15.0)
        sg[kind] = free_sag["sag_mm"]
        w[kind] = lim["weight_kg"]
        sc[kind] = lim["scaled"]
    checks.append(("mini-project: stress-only designs sag 21.8, 25.4, 22.7 mm (Pratt, Howe, Warren), all above the 15 mm limit",
                   abs(sg["pratt"] - 21.80) < 0.005 and abs(sg["howe"] - 25.40) < 0.005 and abs(sg["warren"] - 22.68) < 0.005))
    checks.append(("mini-project: with the 15 mm limit the weights are 111, 135.6 and 115 kg: Pratt is lightest",
                   abs(w["pratt"] - 110.94) < 0.01 and abs(w["howe"] - 135.65) < 0.01 and abs(w["warren"] - 115.05) < 0.01
                   and w["pratt"] < w["warren"] < w["howe"]))
    checks.append(("mini-project: areas are scaled up by 1.45, 1.69 and 1.51 to meet the limit",
                   abs(sc["pratt"] - 1.453) < 0.001 and abs(sc["howe"] - 1.693) < 0.001 and abs(sc["warren"] - 1.512) < 0.001))
    # --- edge cases
    z = L.solve_truss(nodes, members, EA, sup, {2: (0.0, 0.0)})
    checks.append(("edge: zero load gives zero forces and zero deflection", np.allclose(z["forces"], 0.0) and np.allclose(z["u"], 0.0)))
    up = L.solve_truss(nodes, members, EA, sup, {2: (0.0, 10.0)})
    checks.append(("edge: reversing the load reverses every force (tension and compression swap)", np.allclose(up["forces"], -F, atol=1e-9)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
